# 03. Enriquecimento e Modelagem dos Dados

**Objetivo:** estruturar os dados para análise, separando entidades e criando tabelas analíticas adequadas para exploração e visualização no Power BI.

## 1. Leitura da base tratada

Nesta etapa, a base previamente limpa e padronizada é carregada para início do enriquecimento e modelagem.

In [2]:
import pandas as pd
import numpy as np
import re
import os

CAMINHO_ARQUIVO = "../data/tratado/agrofit_limpo.csv"

print("Arquivo existe?", os.path.exists(CAMINHO_ARQUIVO))

df = pd.read_csv(CAMINHO_ARQUIVO, sep=";", encoding="utf-8-sig")

print("Base tratada carregada com sucesso.")
print("Shape da base:", df.shape)

display(df.head())

Arquivo existe? True
Base tratada carregada com sucesso.
Shape da base: (267188, 15)


,nr_registro,marca_comercial,formulacao,ingrediente_ativo,titular_de_registro,classe,modo_de_acao,cultura,praga_nome_cientifico,praga_nome_comum,empresa_pais_tipo,classe_toxicologica,classe_ambiental,organicos,situacao
0,35523,KBR-829M1-02,Nematóides vivos,Heterorhabditis bacteriophora (Nematóides ento...,Koppert do Brasil Holding S.A. - Piracicaba/SP,Agente Biológico de Controle,Não informado,Todas as culturas,Scaptocoris castanea,Percevejo-castanho,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Não Classificado - Produto Não Classificado,Produto Pouco Perigoso ao Meio Ambiente,NAO,True
1,35523,KBR-829M1-02,Nematóides vivos,Heterorhabditis bacteriophora (Nematóides ento...,Koppert do Brasil Holding S.A. - Piracicaba/SP,Agente Biológico de Controle,Não informado,Todas as culturas,Sphenophorus levis,Bicudo da cana-de-açúcar; Gorgulho-da-cana,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Não Classificado - Produto Não Classificado,Produto Pouco Perigoso ao Meio Ambiente,NAO,True
2,35523,KBR-829M1-02,Nematóides vivos,Heterorhabditis bacteriophora (Nematóides ento...,Koppert do Brasil Holding S.A. - Piracicaba/SP,Agente Biológico de Controle,Não informado,Todas as culturas,Spodoptera frugiperda,Lagarta-militar,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Não Classificado - Produto Não Classificado,Produto Pouco Perigoso ao Meio Ambiente,NAO,True
3,33723,KBR-S39M1-02,Nematóides vivos,Steinernema carpocapsae (Nematóides entomopato...,Koppert do Brasil Holding S.A. - Piracicaba/SP,Agente Biológico de Controle,Não informado,Todas as culturas,Bradysia matogrossensis,fungus gnats,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Não Classificado - Produto Não Classificado,Produto Pouco Perigoso ao Meio Ambiente,NAO,True
4,33723,KBR-S39M1-02,Nematóides vivos,Steinernema carpocapsae (Nematóides entomopato...,Koppert do Brasil Holding S.A. - Piracicaba/SP,Agente Biológico de Controle,Não informado,Todas as culturas,Sphenophorus levis,Bicudo da cana-de-açúcar; Gorgulho-da-cana,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Não Classificado - Produto Não Classificado,Produto Pouco Perigoso ao Meio Ambiente,NAO,True


## 2. Entendimento da granularidade

A análise exploratória indicou que a base não está no nível de produto, mas sim no relacionamento entre produto, praga e cultura. Nesta etapa, isso é novamente validado para orientar a modelagem.

In [3]:
produtos_unicos = df["nr_registro"].nunique()

print("Total de linhas:", len(df))
print("Produtos únicos:", produtos_unicos)
print("Média de linhas por produto:", round(len(df) / produtos_unicos, 2))

Total de linhas: 267188
Produtos únicos: 4238
Média de linhas por produto: 63.05


## 3. Criação da dimensão de produtos

Foi criada uma tabela dimensional com uma linha por produto, contendo atributos principais de identificação, classificação e características técnicas.

In [4]:
colunas_dim_produto = [
    "nr_registro",
    "marca_comercial",
    "formulacao",
    "ingrediente_ativo",
    "titular_de_registro",
    "classe",
    "modo_de_acao",
    "classe_toxicologica",
    "classe_ambiental",
    "organicos",
    "situacao"
]

dim_produto = (
    df[colunas_dim_produto]
    .drop_duplicates(subset=["nr_registro"])
    .reset_index(drop=True)
)

print("Shape da dimensão de produto:", dim_produto.shape)
display(dim_produto.head())

Shape da dimensão de produto: (4238, 11)


,nr_registro,marca_comercial,formulacao,ingrediente_ativo,titular_de_registro,classe,modo_de_acao,classe_toxicologica,classe_ambiental,organicos,situacao
0,35523,KBR-829M1-02,Nematóides vivos,Heterorhabditis bacteriophora (Nematóides ento...,Koppert do Brasil Holding S.A. - Piracicaba/SP,Agente Biológico de Controle,Não informado,Não Classificado - Produto Não Classificado,Produto Pouco Perigoso ao Meio Ambiente,NAO,True
1,33723,KBR-S39M1-02,Nematóides vivos,Steinernema carpocapsae (Nematóides entomopato...,Koppert do Brasil Holding S.A. - Piracicaba/SP,Agente Biológico de Controle,Não informado,Não Classificado - Produto Não Classificado,Produto Pouco Perigoso ao Meio Ambiente,NAO,True
2,17622,"1,4 Sight",HN - Concentrado Termo Nebulizável,"1,4 Dimetilnaftaleno (hidrocarbonetos aromátic...",Bem Brasil Alimentos S/A.  Perdizes/MG,Regulador de Crescimento,Não informado,Categoria 4  Produto Pouco Tóxico,Produto Pouco Perigoso ao Meio Ambiente,NAO,True
3,31918,"2,4-D (240) + Picloram (64) SL",SL - Concentrado Solúvel,"2,4-D (ácido ariloxialcanóico) (406 g/L) + pic...",CCAB Agro S.A.  São Paulo,Herbicida,sistêmico,Categoria 5  Produto Improvável de Causar Dan...,Produto Perigoso ao Meio Ambiente,NAO,True
4,13525,"2,4-D + Picloram CAC",SL - Concentrado Solúvel,"2,4-D-trietanolamina (ácido ariloxialcanóico) ...",CAC Quimica do Brasil Ltda,Herbicida,Não informado,Categoria 5  Produto Improvável de Causar Dan...,Produto Perigoso ao Meio Ambiente,NAO,True


## 4. Criação da tabela fato produto × praga × cultura

Foi criada uma tabela relacional contendo as associações entre produtos, culturas e pragas, preservando a granularidade original necessária para análise.

In [5]:
colunas_fato_produto_praga = [
    "nr_registro",
    "cultura",
    "praga_nome_cientifico",
    "praga_nome_comum"
]

fato_produto_praga = (
    df[colunas_fato_produto_praga]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Shape da fato produto x praga:", fato_produto_praga.shape)
display(fato_produto_praga.head())

Shape da fato produto x praga: (267188, 4)


,nr_registro,cultura,praga_nome_cientifico,praga_nome_comum
0,35523,Todas as culturas,Scaptocoris castanea,Percevejo-castanho
1,35523,Todas as culturas,Sphenophorus levis,Bicudo da cana-de-açúcar; Gorgulho-da-cana
2,35523,Todas as culturas,Spodoptera frugiperda,Lagarta-militar
3,33723,Todas as culturas,Bradysia matogrossensis,fungus gnats
4,33723,Todas as culturas,Sphenophorus levis,Bicudo da cana-de-açúcar; Gorgulho-da-cana


## 5. Enriquecimento de informações de empresa

A coluna `empresa_pais_tipo` concentra múltiplas informações em um único campo. Nesta etapa, são criadas colunas auxiliares para facilitar análises futuras sobre localização e composição empresarial.

In [6]:
def extrair_estado(texto):
    if pd.isna(texto) or texto == "Não informado":
        return np.nan

    match = re.search(r"/([A-Z]{2})", texto)
    return match.group(1) if match else np.nan


def extrair_pais_mencionado(texto):
    if pd.isna(texto) or texto == "Não informado":
        return np.nan

    match = re.search(r"<([^>]+)>", texto)
    return match.group(1).strip() if match else np.nan


df["empresa_estado"] = df["empresa_pais_tipo"].apply(extrair_estado)
df["empresa_pais_mencionado"] = df["empresa_pais_tipo"].apply(extrair_pais_mencionado)

display(
    df[
        [
            "empresa_pais_tipo",
            "empresa_estado",
            "empresa_pais_mencionado"
        ]
    ].head(10)
)

,empresa_pais_tipo,empresa_estado,empresa_pais_mencionado
0,(Koppert do Brasil Holding S.A. - Piracicaba/S...,SP,BRASIL
1,(Koppert do Brasil Holding S.A. - Piracicaba/S...,SP,BRASIL
2,(Koppert do Brasil Holding S.A. - Piracicaba/S...,SP,BRASIL
3,(Koppert do Brasil Holding S.A. - Piracicaba/S...,SP,BRASIL
4,(Koppert do Brasil Holding S.A. - Piracicaba/S...,SP,BRASIL
5,(Koppert do Brasil Holding S.A. - Piracicaba/S...,SP,BRASIL
6,"(Sinochem Hebei Fuheng Co., Ltd. - Hebei<CHINA...",NaN,"CHINA, REPUBLICA POPULAR"
7,(Tagma Brasil Indústria e Comércio de Produtos...,NaN,BRASIL
8,(Tagma Brasil Indústria e Comércio de Produtos...,NaN,BRASIL
9,(Tagma Brasil Indústria e Comércio de Produtos...,NaN,BRASIL


## 6. Classificação simplificada do perfil da empresa

A coluna `empresa_pais_tipo` apresenta estrutura semiestruturada e múltiplas entidades em um único campo.  
Para fins analíticos, foi criada uma classificação simplificada que distingue empresas com atuação exclusivamente nacional daquelas com presença internacional.

In [7]:
def classificar_tipo_empresa(texto):
    if pd.isna(texto) or texto == "Não informado":
        return "Não informado"
    
    texto_upper = str(texto).upper()

    paises_estrangeiros = [
        "CHINA",
        "ESTADOS UNIDOS",
        "EUA",
        "MÉXICO",
        "MEXICO",
        "ARGENTINA",
        "HOLANDA",
        "PAÍSES BAIXOS",
        "PAISES BAIXOS",
        "FRANÇA",
        "FRANCA",
        "ÍNDIA",
        "INDIA",
        "ISRAEL",
        "JAPÃO",
        "JAPAO",
        "ALEMANHA",
        "BÉLGICA",
        "BELGICA",
        "PARAGUAI",
        "PORTUGAL",
        "ESPANHA",
        "ÁUSTRIA",
        "AUSTRIA",
        "BÉLGICA",
        "BELGICA"
    ]

    tem_brasil = "BRASIL" in texto_upper
    tem_estrangeiro = any(pais in texto_upper for pais in paises_estrangeiros)

    if tem_estrangeiro:
        return "Com presença internacional"
    
    if tem_brasil:
        return "Somente nacional"
    
    return "Não identificado"


df["tipo_empresa"] = df["empresa_pais_tipo"].apply(classificar_tipo_empresa)

display(df[["empresa_pais_tipo", "tipo_empresa"]].head(10))

,empresa_pais_tipo,tipo_empresa
0,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Com presença internacional
1,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Com presença internacional
2,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Com presença internacional
3,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Com presença internacional
4,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Com presença internacional
5,(Koppert do Brasil Holding S.A. - Piracicaba/S...,Com presença internacional
6,"(Sinochem Hebei Fuheng Co., Ltd. - Hebei<CHINA...",Com presença internacional
7,(Tagma Brasil Indústria e Comércio de Produtos...,Com presença internacional
8,(Tagma Brasil Indústria e Comércio de Produtos...,Com presença internacional
9,(Tagma Brasil Indústria e Comércio de Produtos...,Com presença internacional


## 7. Distribuição do tipo de empresa na base

A classificação criada permite avaliar, de forma exploratória, a presença de registros com atuação exclusivamente nacional e aqueles com participação internacional na cadeia descrita em `empresa_pais_tipo`.

In [17]:
dist_tipo_empresa = (
    df["tipo_empresa"]
    .value_counts()
    .reset_index()
)

dist_tipo_empresa.columns = ["tipo_empresa", "qtd_registros"]

display(dist_tipo_empresa)

,tipo_empresa,qtd_registros
0,Com presença internacional,241473
1,Somente nacional,25111
2,Não identificado,508
3,Não informado,96


## 8. Criação da dimensão auxiliar de empresas

Foi criada uma dimensão auxiliar com informações de empresa e localização, incluindo a classificação de perfil de atuação.  
Essa tabela permanece útil para análises exploratórias, mas não será a dimensão principal do modelo analítico no Power BI.

In [18]:
colunas_dim_empresa = [
    "titular_de_registro",
    "empresa_pais_tipo",
    "empresa_estado",
    "empresa_pais_mencionado",
    "tipo_empresa"
]

dim_empresa = (
    df[colunas_dim_empresa]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Shape da dimensão de empresa:", dim_empresa.shape)
display(dim_empresa.head())

Shape da dimensão de empresa: (3367, 5)


,titular_de_registro,empresa_pais_tipo,empresa_estado,empresa_pais_mencionado,tipo_empresa
0,Koppert do Brasil Holding S.A. - Piracicaba/SP,(Koppert do Brasil Holding S.A. - Piracicaba/S...,SP,BRASIL,Com presença internacional
1,Koppert do Brasil Holding S.A. - Piracicaba/SP,(Koppert do Brasil Holding S.A. - Piracicaba/S...,SP,BRASIL,Com presença internacional
2,Bem Brasil Alimentos S/A.  Perdizes/MG,"(Sinochem Hebei Fuheng Co., Ltd. - Hebei<CHINA...",NaN,"CHINA, REPUBLICA POPULAR",Com presença internacional
3,CCAB Agro S.A.  São Paulo,(Tagma Brasil Indústria e Comércio de Produtos...,NaN,BRASIL,Com presença internacional
4,CAC Quimica do Brasil Ltda,"(Jiangxi Tianyu Chemical Co., Ltd.<CHINA, REPU...",SP,"CHINA, REPUBLICA POPULAR",Com presença internacional


## 9. Criação da dimensão de titulares

Foi criada uma dimensão auxiliar contendo uma linha por titular de registro.

Essa estrutura tem como objetivo viabilizar o relacionamento entre produtos e informações empresariais no Power BI, já que a coluna `titular_de_registro` não é única na base de produtos nem na dimensão auxiliar de empresas.

Dessa forma, a `dim_titular` passa a ser a dimensão principal para análises relacionadas aos titulares de registro.

In [19]:
dim_titular = (
    df[
        [
            "titular_de_registro",
            "tipo_empresa"
        ]
    ]
    .drop_duplicates(subset=["titular_de_registro"])
    .reset_index(drop=True)
)

print("Shape da dimensão de titulares:", dim_titular.shape)
display(dim_titular.head())

Shape da dimensão de titulares: (343, 2)


,titular_de_registro,tipo_empresa
0,Koppert do Brasil Holding S.A. - Piracicaba/SP,Com presença internacional
1,Bem Brasil Alimentos S/A.  Perdizes/MG,Com presença internacional
2,CCAB Agro S.A.  São Paulo,Com presença internacional
3,CAC Quimica do Brasil Ltda,Com presença internacional
4,Inovatis Agronegocios Importação e Exportação ...,Com presença internacional


## 10. Distribuição do tipo de empresa (nível empresa)

Nesta análise, considera-se a dimensão de titulares, contabilizando apenas empresas únicas.

In [20]:
dist_tipo_empresa_titular = (
    dim_titular["tipo_empresa"]
    .value_counts()
    .reset_index()
)

dist_tipo_empresa_titular.columns = ["tipo_empresa", "qtd_empresas"]

display(dist_tipo_empresa_titular)

,tipo_empresa,qtd_empresas
0,Somente nacional,178
1,Com presença internacional,157
2,Não informado,5
3,Não identificado,3


## 11. Interpretação inicial

Observa-se predominância de registros e também de empresas classificadas como "Com presença internacional".

Entretanto, essa classificação deve ser analisada com cautela.

A variável `tipo_empresa` não representa exclusivamente a origem da empresa titular, mas sim a presença de diferentes países na composição do registro, incluindo formuladores, fabricantes ou parceiros.

Dessa forma, uma empresa brasileira pode ser classificada como "Com presença internacional" caso haja participação de entidades estrangeiras no processo produtivo.

Esse comportamento evidencia a complexidade da cadeia de produção de defensivos agrícolas e reforça a importância de distinguir entre origem da empresa e estrutura da cadeia produtiva.

## 12. Validação das tabelas geradas

Após a criação da dimensão de titulares, são verificadas novamente as dimensões das tabelas analíticas finais.

In [21]:
print("Base tratada original:", df.shape)
print("dim_produto:", dim_produto.shape)
print("fato_produto_praga:", fato_produto_praga.shape)
print("dim_empresa:", dim_empresa.shape)
print("dim_titular:", dim_titular.shape)

Base tratada original: (267188, 18)
dim_produto: (4238, 11)
fato_produto_praga: (267188, 4)
dim_empresa: (3367, 5)
dim_titular: (343, 2)


## 13. Exportação das tabelas analíticas

As tabelas geradas nesta etapa são exportadas para uso posterior no Power BI.

In [22]:
CAMINHO_DIM_PRODUTO = "../data/tratado/dim_produto.csv"
CAMINHO_FATO_PRODUTO_PRAGA = "../data/tratado/fato_produto_praga.csv"
CAMINHO_DIM_EMPRESA = "../data/tratado/dim_empresa.csv"
CAMINHO_DIM_TITULAR = "../data/tratado/dim_titular.csv"

dim_produto.to_csv(CAMINHO_DIM_PRODUTO, index=False, sep=";", encoding="utf-8-sig")
fato_produto_praga.to_csv(CAMINHO_FATO_PRODUTO_PRAGA, index=False, sep=";", encoding="utf-8-sig")
dim_empresa.to_csv(CAMINHO_DIM_EMPRESA, index=False, sep=";", encoding="utf-8-sig")
dim_titular.to_csv(CAMINHO_DIM_TITULAR, index=False, sep=";", encoding="utf-8-sig")

print("Arquivos exportados com sucesso:")
print(CAMINHO_DIM_PRODUTO)
print(CAMINHO_FATO_PRODUTO_PRAGA)
print(CAMINHO_DIM_EMPRESA)
print(CAMINHO_DIM_TITULAR)

Arquivos exportados com sucesso:
../data/tratado/dim_produto.csv
../data/tratado/fato_produto_praga.csv
../data/tratado/dim_empresa.csv
../data/tratado/dim_titular.csv


## 14. Principais ações realizadas

Nesta etapa, foram executadas as seguintes ações:

- Validação da granularidade da base
- Criação de uma dimensão de produtos
- Criação de uma tabela fato produto × praga × cultura
- Extração de informações auxiliares de empresa/localização
- Classificação simplificada do perfil de atuação das empresas
- Criação de uma dimensão auxiliar de empresas
- Criação de uma dimensão de titulares para suporte à modelagem no Power BI
- Exportação das tabelas analíticas para consumo no dashboard

> A criação da `dim_titular` foi necessária para garantir integridade na modelagem, uma vez que `titular_de_registro` não é uma chave única em `dim_produto` nem em `dim_empresa`.